# InvertedPendulum Horizon-to-Failure World Model Homework

This notebook runs the official homework workflow: install dependencies, generate MuJoCo ground-truth windows, train the one-step baseline, train the student world model, evaluate H80 survival horizon, and plot diagnostics.

The assignment trains **one dynamics world model**. It does not train a policy, does not use reward, and does not use actor-critic updates.

## 1. Configure Repository

If you are running this notebook inside a local clone, the setup cell will reuse the current repository. Otherwise, set `COURSE_REPO_URL` to your fork and the notebook will clone it.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_WorldModel-Homework.git"
COURSE_REPO_BRANCH = "main"
SMOKE_RUN = True  # Set False for the full public profile.

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)

def find_existing_repo(start: Path):
    for path in [start, *start.parents]:
        if (path / "wm_hw").is_dir() and (path / "configs" / "colab.yaml").exists():
            return path
    return None

def default_clone_dir():
    content = Path("/content")
    if content.exists() and os.access(content, os.W_OK):
        return content / "wm_inverted_pendulum_hw"
    return Path.home() / "wm_inverted_pendulum_hw"

repo = find_existing_repo(Path.cwd())
if repo is None:
    repo = default_clone_dir()
    if not (repo / ".git").exists():
        run(["git", "clone", "--branch", COURSE_REPO_BRANCH, COURSE_REPO_URL, repo])
    else:
        print(f"Using existing repo at {repo}")

COURSE_REPO_DIR = repo.resolve()
print("COURSE_REPO_DIR =", COURSE_REPO_DIR)

## 2. Install Dependencies And Verify MuJoCo

In [ ]:
%cd {COURSE_REPO_DIR}
!{sys.executable} -m pip install -q -U pip setuptools wheel
!{sys.executable} -m pip install -q -r requirements.txt

import gymnasium as gym
import torch

env = gym.make("InvertedPendulum-v5", reset_noise_scale=0.01)
obs, info = env.reset(seed=0)
print("obs shape:", obs.shape)
print("action shape:", env.action_space.shape)
print("action range:", env.action_space.low, env.action_space.high)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
env.close()

## 3. Run Tests

These tests check the environment, dataset schema, no-leak open-loop rollout, differentiable loss, failure metric, and train/eval smoke path.

In [ ]:
!{sys.executable} -m pytest -q

## 4. Generate MuJoCo Ground-Truth Windows

In [ ]:
DATASET_DIR = "data/public_smoke" if SMOKE_RUN else "data/public"
smoke_flag = "--smoke" if SMOKE_RUN else ""
!{sys.executable} -m wm_hw.dataset --config configs/colab.yaml --output-dir {DATASET_DIR} {smoke_flag}

## 5. Train The Locked One-Step Baseline

In [ ]:
!{sys.executable} -m wm_hw.train --config configs/baseline.yaml --model baseline --dataset-dir {DATASET_DIR} --output-dir artifacts/baseline {smoke_flag}

## 6. Train The Student World Model

In [ ]:
!{sys.executable} -m wm_hw.train --config configs/student.yaml --model student --dataset-dir {DATASET_DIR} --output-dir artifacts/student {smoke_flag}

## 7. Evaluate Test And OOD H80

In [ ]:
!{sys.executable} -m wm_hw.eval_horizon --checkpoint-dir artifacts/student/best_checkpoint --dataset-dir {DATASET_DIR} --split test --output-dir artifacts/student/eval_test
!{sys.executable} -m wm_hw.eval_horizon --checkpoint-dir artifacts/student/best_checkpoint --dataset-dir {DATASET_DIR} --split ood --output-dir artifacts/student/eval_ood

## 8. Plot Diagnostics

In [ ]:
!{sys.executable} -m wm_hw.plotting --eval-dir artifacts/student/eval_test --output-dir artifacts/student/plots

import json
from IPython.display import Image, display

with open("artifacts/student/eval_test/metrics.json", "r", encoding="utf-8") as f:
    test_metrics = json.load(f)
with open("artifacts/student/eval_ood/metrics.json", "r", encoding="utf-8") as f:
    ood_metrics = json.load(f)

print("test H80:", test_metrics["H80"])
print("ood H80:", ood_metrics["H80"])
display(Image("artifacts/student/plots/survival_curve.png"))
display(Image("artifacts/student/plots/rollout_comparison.png"))